<a href="https://colab.research.google.com/github/Mudith-cloud/NLP-experiments/blob/main/CSL448_Experiment_6_Ngram_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSL448 – Computational Linguistics and NLP
## Experiment 6 – N-gram Language Models

**Aim:** Build and evaluate unigram, bigram, and trigram language models using Laplace smoothing and compute perplexity on test data.

In [ ]:
# Install and import required libraries
!pip -q install nltk spacy

import math
import re
from collections import Counter
import nltk
import spacy
import pandas as pd

nltk.download('punkt')
nltk.download('punkt_tab')

print("NLTK and spaCy imported successfully.")

## 1. Training and test data

In [ ]:
train_text = """
Natural language processing is a field of artificial intelligence.
Language models learn patterns from text.
A language model assigns probabilities to sequences of words.
N gram models are simple statistical language models.
Unigram models consider one word at a time.
Bigram models consider two consecutive words.
Trigram models consider three consecutive words.
Smoothing helps language models handle unseen sequences.
Natural language processing is useful for search and translation.
Language models are useful in many NLP applications.
"""

test_text = """
Language models learn useful patterns.
Bigram models assign probabilities to word sequences.
Smoothing helps with unseen words.
Natural language processing has many applications.
"""

print("Training sentences:", len(nltk.sent_tokenize(train_text)))
print("Test sentences:", len(nltk.sent_tokenize(test_text)))

## 2. Tokenization

In [ ]:
def tokenize(text):
    return [w.lower() for w in nltk.word_tokenize(text)
            if re.fullmatch(r"[a-zA-Z]+", w)]

def sentence_tokens(text):
    result = []
    for sentence in nltk.sent_tokenize(text):
        words = tokenize(sentence)
        if words:
            result.append(["<s>"] + words + ["</s>"])
    return result

train_sentences = sentence_tokens(train_text)
test_sentences = sentence_tokens(test_text)

print("Example tokenized sentence:")
print(train_sentences[0])

## 3. Generate unigram, bigram and trigram counts

In [ ]:
def get_ngram_counts(sentences, n):
    counts = Counter()
    for sent in sentences:
        for i in range(len(sent) - n + 1):
            counts[tuple(sent[i:i+n])] += 1
    return counts

unigram_counts = get_ngram_counts(train_sentences, 1)
bigram_counts = get_ngram_counts(train_sentences, 2)
trigram_counts = get_ngram_counts(train_sentences, 3)

print("Top 10 unigrams:", unigram_counts.most_common(10))
print("Top 10 bigrams:", bigram_counts.most_common(10))
print("Top 10 trigrams:", trigram_counts.most_common(10))

## 4. Laplace smoothing

Laplace/add-one smoothing adds 1 to every n-gram count.

- Unigram: `(count(w) + 1) / (N + V)`
- Bigram: `(count(w1,w2) + 1) / (count(w1) + V)`
- Trigram: `(count(w1,w2,w3) + 1) / (count(w1,w2) + V)`

In [ ]:
vocab = set(tokenize(train_text))
vocab.update(["<UNK>", "</s>", "<s>"])
V = len(vocab)
N = sum(unigram_counts.values())

def map_unknown(word):
    return word if word in vocab else "<UNK>"

def unigram_prob(word):
    word = map_unknown(word)
    return (unigram_counts.get((word,), 0) + 1) / (N + V)

def bigram_prob(w1, w2):
    w1, w2 = map_unknown(w1), map_unknown(w2)
    return (bigram_counts.get((w1, w2), 0) + 1) / (
        unigram_counts.get((w1,), 0) + V
    )

def trigram_prob(w1, w2, w3):
    w1, w2, w3 = map_unknown(w1), map_unknown(w2), map_unknown(w3)
    return (trigram_counts.get((w1, w2, w3), 0) + 1) / (
        bigram_counts.get((w1, w2), 0) + V
    )

print("Vocabulary size:", V)
print("P(language) =", unigram_prob("language"))
print("P(models | language) =", bigram_prob("language", "models"))
print("P(models | language, processing) =",
      trigram_prob("language", "processing", "models"))

## 5. Compute perplexity

In [ ]:
def corpus_perplexity(text, n):
    total_log_prob = 0
    total_tokens = 0

    for sentence in nltk.sent_tokenize(text):
        words = [map_unknown(w) for w in tokenize(sentence)]

        if n == 1:
            tokens = words + ["</s>"]
            for w in tokens:
                total_log_prob += math.log(unigram_prob(w))
            total_tokens += len(tokens)

        elif n == 2:
            tokens = ["<s>"] + words + ["</s>"]
            for i in range(1, len(tokens)):
                total_log_prob += math.log(
                    bigram_prob(tokens[i-1], tokens[i])
                )
            total_tokens += len(tokens) - 1

        elif n == 3:
            tokens = ["<s>", "<s>"] + words + ["</s>"]
            for i in range(2, len(tokens)):
                total_log_prob += math.log(
                    trigram_prob(tokens[i-2], tokens[i-1], tokens[i])
                )
            total_tokens += len(tokens) - 2

    return math.exp(-total_log_prob / total_tokens)

results = {}
for n in [1, 2, 3]:
    results[f"{n}-gram"] = corpus_perplexity(test_text, n)

comparison = pd.DataFrame({
    "Model": results.keys(),
    "Perplexity": results.values()
})

comparison

## 6. Result and conclusion

**Perplexity:** Lower values indicate that the model predicts the test corpus better.

**Conclusion:** Unigram, bigram and trigram language models were successfully implemented using Laplace smoothing. Perplexity was calculated on unseen test data to compare their performance.